# 1. Importação dos Dados e Bibliotecas.

Nesta etapa, a biblioteca Pandas é importada e o arquivo de assinaturas é carregado e convertido em um DataFrame.

In [1]:
from pathlib import Path
import sys

RAIZ    = Path.cwd().parent
BRONZE  = RAIZ / "data" / "Bronze"
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

assert BRONZE.exists(), f"Pasta não encontrada: {BRONZE}. Abra o notebook a partir da pasta notebooks/."

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)
sys.path.append(str(RAIZ))

## Importação de Bibliotecas:
import pandas as pd
import numpy as np

## Importando funções da pasta src.
from src.datas import converter_data

df_assinatura = pd.read_csv(BRONZE / "assinaturas.csv")
print(df_assinatura.info())
original = df_assinatura.copy()


<class 'pandas.DataFrame'>
RangeIndex: 3010 entries, 0 to 3009
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_assinatura      3010 non-null   int64
 1   id_cliente         3010 non-null   int64
 2   id_plano           3010 non-null   int64
 3   data_inicio        3010 non-null   str  
 4   data_cancelamento  871 non-null    str  
 5   status             3010 non-null   str  
dtypes: int64(3), str(3)
memory usage: 197.6 KB
None


# 2. Tratamento e Análise de Dados:

Nesta etapa, é realizado o tratamento completo dos dados, incluindo a validação de registros duplicados, a alteração de status e a conversão de tipos (types), com base na análise dos seguintes pontos críticos:

Verificação de duplicatas: Validar a existência de IDs principais duplicados.
Limpeza de strings: Remover os espaços em branco de todas as colunas.
Validação numérica: Verificar a presença de IDs negativos nas colunas 1 e 2.
Conversão de tipos (Datas): Ajustar o formato/tipo (type) da coluna de data e da data de cancelamento.
Consistência temporal: Garantir que a data de cancelamento não seja anterior à data de início.
Validação de status: Analisar os dados presentes na coluna de status e atualizar automaticamente o valor do status caso a data de cancelamento esteja preenchida.

## 2.1 Tratamento Coluna id_assinaturas:

Nesta etapa, está sendo realizada a análise a respeito se existem dados duplicados, onde devemos considerar como verdadeiro caso seja maior que zero, além disso, foi realizado o filtro para análise dos IDs duplicados, a fim de verificar se trata-se de uma linha duplicada ou não, para seguir com a devida tratativa.

Nesta etapa, irei apagar as segundas linhas que se duplicam usando como referência o id_assinatura, que por regra, não deve se repetir nesse arquivo e após isso, será realizada a validação, se consta IDs, duplicados, para que o tratamento esteja correto, sempre retornara 0, caso contrário, ainda existira linhas com id assinatura duplicado.

In [2]:
# 1. Validação: de IDs duplicidados
total_duplicados = df_assinatura['id_assinatura'].duplicated().sum()
print(f"Total de IDs duplicados: {total_duplicados}")

# 2. Tratamento: Deletando linhas duplicadas.
df_assinatura = df_assinatura.drop_duplicates(subset=['id_assinatura'], keep='first')

# 3. Validaçao: Validando resultado antes do tratamento.
total_duplicados = df_assinatura['id_assinatura'].duplicated().sum()
print(f"Total de IDs duplicados: {total_duplicados}")

# 3.1 Validaçao: Validando resultado pós do tratamento.
df_assinatura = df_assinatura.drop_duplicates(subset=['id_assinatura'], keep='first')
total_duplicados = df_assinatura['id_assinatura'].duplicated().sum()
print(f"Total de IDs duplicados: {total_duplicados}")


Total de IDs duplicados: 10
Total de IDs duplicados: 0
Total de IDs duplicados: 0


## 2.2 Tratamento da Coluna ID_CLIENTE:
### Validação dos números dos IDs

Nesta etapa, apenas será verificado se existem IDs inferiores a 0, vale-se destacar que o arquivo de cliente, não possui IDs negativos, se for identificado algum ID negativo, trata-se de um erro. Além disso, os dados dessa coluna podem se repetir.

In [3]:
# 1. Validação: Validando se existe valores negativos.
validacao_negativos = (df_assinatura['id_cliente'] < 0).sum()
print(f'Valores negativos encontrados, é igual a: {validacao_negativos}')

Valores negativos encontrados, é igual a: 0


## 2.3 Tratamento da Coluna ID_PLANO:

Nesta etapa, apenas será verificado se existem IDs inferiores a 0, vale-se destacar que no arquivo plano, não possui IDs negativos, se for identificado algum ID negativo, trata-se de um erro. Além disso, os dados dessa coluna podem se repetir.

In [4]:
# 1. Validação: Validando se existe valores negativos.
validacao_negativos = (df_assinatura['id_plano'] < 0).sum()
print(f'Valores negativos encontrados, é igual a: {validacao_negativos}')

Valores negativos encontrados, é igual a: 0


## 2.4 Tratamento da Coluna Data_Inicio e Data_Cancelamento
### Transformação do Type:

Nesta etapa, a coluna data_inicio e data_cancelamento é convertida do formato texto (string) para o tipo datetime. Como os dados apresentam múltiplos formatos de data, será realizada a padronização de todos os registros para o formato padrão US. Além disso, será gerada uma coluna de reporte data cancelamento, para futuras análises de datas divergentes preenchidas.

In [5]:
# 1. Tratamento de Dados: Transformando as colunas de data em datetime (formato ISO).
df_assinatura['data_inicio'] = converter_data(df_assinatura['data_inicio'])
df_assinatura['data_cancelamento'] = converter_data(df_assinatura['data_cancelamento'])

# 2. Validação de Dados: Validando dados Pós Tratamento:
print("Tipo data_inicio:", df_assinatura['data_inicio'].dtype)
print("Tipo data_cancelamento:", df_assinatura['data_cancelamento'].dtype)
print("Datas de início vazias:", df_assinatura['data_inicio'].isna().sum())

Tipo data_inicio: datetime64[us]
Tipo data_cancelamento: datetime64[us]
Datas de início vazias: 0


In [6]:
# 1. Criação da coluna de sinalização (reporte_data_cancelamento)
# Inicializamos a coluna preenchendo todos os valores com "OK"
df_assinatura['reporte_data_cancelamento'] = 'OK'

# 2. Identifica onde a data de cancelamento é INFERIOR (menor) que a data de início
# (Apenas linhas onde ambas as datas estão preenchidas e a condição é verdadeira)
condicao_divergencia = df_assinatura['data_cancelamento'] < df_assinatura['data_inicio']
print(f'Existem valores divergentes: {condicao_divergencia}')

# 3. Caso a divergência seja verdadeira:
# A) Altera a data de cancelamento para ser igual à data de início
df_assinatura.loc[condicao_divergencia, 'data_cancelamento'] = df_assinatura['data_inicio']

# B) Sinaliza na nova coluna 'reporte_data_cancelamento' que houve um erro/divergência para a equipe investigar
df_assinatura.loc[condicao_divergencia, 'reporte_data_cancelamento'] = 'Analisar'
print(df_assinatura.info())

Existem valores divergentes: 0       False
1       False
2       False
3       False
4       False
        ...  
2995    False
2996    False
2997    False
2998    False
2999    False
Length: 3000, dtype: bool
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id_assinatura              3000 non-null   int64         
 1   id_cliente                 3000 non-null   int64         
 2   id_plano                   3000 non-null   int64         
 3   data_inicio                3000 non-null   datetime64[us]
 4   data_cancelamento          867 non-null    datetime64[us]
 5   status                     3000 non-null   str           
 6   reporte_data_cancelamento  3000 non-null   str           
dtypes: datetime64[us](2), int64(3), str(2)
memory usage: 188.5 KB
None


## 2.5 Tratamento da Coluna Status:

Nesta etapa, os valores da coluna status são analisados para identificar inconsistências e padronizar os termos. Além disso, a coluna foi automatizada por meio de uma regra condicional: se houver uma data de cancelamento preenchida, o status é definido como "Cancelado"; caso contrário, o status permanece como "Ativo".

In [7]:
# Se a data de cancelamento estiver preenchida, o status vira 'Cancelado'
df_assinatura.loc[df_assinatura['data_cancelamento'].notna(), 'status'] = 'Cancelado'

# Se a data de cancelamento estiver vazia, o status vira 'Ativo'
df_assinatura.loc[df_assinatura['data_cancelamento'].isna(), 'status'] = 'Ativo'

# 3. Data Quality & Validação.

Nesta etapa, o DataFrame tratado é submetido a testes de validação antes de ser promovido para a Staging (Preparação de Dados).

In [8]:
resultado_qualidade= {}

# 1. Validaçaõ de IDs não nulos:
resultado_qualidade['ID não nulo'] = (
    df_assinatura['id_assinatura'].isnull().sum() == 0
)

# 2. Validação a fim de identificar se existem IDs duplicados
resultado_qualidade['ID único'] = (
    df_assinatura['id_assinatura'].duplicated().sum() == 0
)

# 3. Validação se existem linhas duplicadas
resultado_qualidade['Sem duplicatas completas'] = (
    df_assinatura.duplicated().sum() == 0
)

# 4. Validação se todas as linhas da coluna id_assinatura estão preenchidas
resultado_qualidade['Linhas preenchidas id_assinatura'] = (
    df_assinatura['id_assinatura'].notna().all()
)

# 5. Validação se a coluna data_inicio está com o formato correto de type
resultado_qualidade['Tipo de dado da coluna data_inicio'] = (
    pd.api.types.is_datetime64_any_dtype(df_assinatura['data_inicio'])
)

# 6. Validação se a coluna data_cancelamento está com o formato correto de type
resultado_qualidade['Tipo de dado da coluna data_cancelamento'] = (
    pd.api.types.is_datetime64_any_dtype(df_assinatura['data_cancelamento'])
)

# 7. Validação valores de IDs negativos
resultado_qualidade['ID cliente válido'] = (
    (df_assinatura['id_cliente'] >= 0).all()
)

# 8. Validação Data Inicio - valores divergentes
resultado_qualidade['Data início válida'] = (
    df_assinatura['data_inicio'].notna().all()
)

# 4. Data Quality: Homologação e Envio para a Camada Staging (Preparação de Dados)

Nesta etapa, o console exibe o resultado das validações realizadas:

Aprovado (True em todos os testes): O DataFrame é liberado e enviado para a camada Staging (Preparação de Dados)
Reprovado (qualquer valor False): O processo é bloqueado, exigindo a correção dos dados antes de avançar.

In [9]:
df_qualidade = pd.DataFrame(
    list(resultado_qualidade.items()),
    columns=['Regra', 'Aprovado']
)

df_qualidade['Status'] = df_qualidade['Aprovado'].map({
    True: 'APROVADO',
    False: 'REPROVADO'
})

display(df_qualidade)

todos_testes_ok = df_qualidade['Aprovado'].all()
caminho_saida =  (STAGING / "dim_assinatura.csv")


if todos_testes_ok:
    print("Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging.")
    df_assinatura.to_csv(
            caminho_saida,
            index=False,
            sep=';',
            encoding='utf-8-sig'
        )
else:
    print("Alerta: O arquivo foi impedido de subir para a camada Staging. Existem regras reprovadas.")
    raise ValueError("Falha na validação de qualidade dos dados. Processo interrompido.")

,Regra,Aprovado,Status
0,ID não nulo,True,APROVADO
1,ID único,True,APROVADO
2,Sem duplicatas completas,True,APROVADO
3,Linhas preenchidas id_assinatura,True,APROVADO
4,Tipo de dado da coluna data_inicio,True,APROVADO
5,Tipo de dado da coluna data_cancelamento,True,APROVADO
6,ID cliente válido,True,APROVADO
7,Data início válida,True,APROVADO


Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Staging.
